---
# Regression Trees with CART (DecisionTreeRegressor)

This notebook demonstrates how to train and evaluate a **regression tree** (CART-style) on a real-world dataset using common Python libraries.

We'll:
- load a public regression dataset,
- split into train/validation/test sets,
- fit a decision tree regressor,
- tune a few hyperparameters to control overfitting,
- evaluate performance (RMSE / $R^2$),
- inspect feature importances and model behavior.

---

---
## 1) Imports and setup
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

---
## 2) Load dataset (California Housing)

We use the **California Housing** dataset bundled with scikit-learn.  
The target is the median house value (in hundreds of thousands of dollars).

---

In [ ]:
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

X.head(), y.head()

(   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
 0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
 1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
 2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
 3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
 4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   
 
    Longitude  
 0    -122.23  
 1    -122.22  
 2    -122.24  
 3    -122.25  
 4    -122.25  ,
 0    4.526
 1    3.585
 2    3.521
 3    3.413
 4    3.422
 Name: MedHouseVal, dtype: float64)

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Feature columns:", list(X.columns))

X shape: (20640, 8)
y shape: (20640,)
Feature columns: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


---
## 3) Train / validation / test split

A decision tree can overfit very easily, so a validation set is helpful for choosing
hyperparameters (like `max_depth`).

---

In [ ]:
# First: split off a test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Second: split train/val
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=RANDOM_STATE
)
# 0.25 of 0.8 -> 0.2, so: train=0.6, val=0.2, test=0.2

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (12384, 8) Val: (4128, 8) Test: (4128, 8)


---
## 4) Baseline regression tree

We start with a default-ish decision tree and evaluate it on train/val/test.
Typically you'll see very low training error and noticeably worse validation/test error.

---

In [ ]:
def regression_report(y_true, y_pred, name=""):
    rmse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"split": name, "rmse": rmse, "mae": mae, "r2": r2}

baseline = DecisionTreeRegressor(random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)

pred_train = baseline.predict(X_train)
pred_val = baseline.predict(X_val)
pred_test = baseline.predict(X_test)

results = [
    regression_report(y_train, pred_train, "train"),
    regression_report(y_val, pred_val, "val"),
    regression_report(y_test, pred_test, "test"),
]
pd.DataFrame(results)

TypeError: got an unexpected keyword argument 'squared'

---
### Quick diagnostic plots

We'll visualize how predictions compare to the truth (on the test split) and
look at the distribution of residuals.

---

In [ ]:
residuals = y_test - pred_test

plt.figure()
plt.scatter(y_test, pred_test, s=10, alpha=0.4)
plt.xlabel("True target")
plt.ylabel("Predicted target")
plt.title("Baseline tree: true vs predicted (test)")
plt.show()

plt.figure()
plt.hist(residuals, bins=50)
plt.xlabel("Residual (y_true - y_pred)")
plt.ylabel("Count")
plt.title("Baseline tree residuals (test)")
plt.show()

---
## 5) Controlling overfitting with `max_depth`

A deeper tree can memorize training data. We'll tune `max_depth` using the validation split.

Notes:
- smaller depth → higher bias, lower variance
- larger depth → lower bias, higher variance (more overfit risk)

---

In [ ]:
depth_grid = list(range(1, 31))
rows = []

for depth in depth_grid:
    model = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    rows.append({
        "max_depth": depth,
        "train_rmse": mean_squared_error(y_train, train_pred, squared=False),
        "val_rmse": mean_squared_error(y_val, val_pred, squared=False),
        "train_r2": r2_score(y_train, train_pred),
        "val_r2": r2_score(y_val, val_pred),
    })

grid_df = pd.DataFrame(rows)
grid_df.head()

In [ ]:
plt.figure()
plt.plot(grid_df["max_depth"], grid_df["train_rmse"], marker="o", label="train RMSE")
plt.plot(grid_df["max_depth"], grid_df["val_rmse"], marker="o", label="val RMSE")
plt.xlabel("max_depth")
plt.ylabel("RMSE")
plt.title("Depth sweep: train vs val RMSE")
plt.legend()
plt.show()

best_row = grid_df.loc[grid_df["val_rmse"].idxmin()]
best_depth = int(best_row["max_depth"])
best_row, best_depth

---
## 6) Train the tuned tree and evaluate on test

Now we retrain using **train + validation** (more data) at the chosen depth,
and evaluate once on the held-out test set.

---

In [ ]:
X_train_full = pd.concat([X_train, X_val], axis=0)
y_train_full = pd.concat([y_train, y_val], axis=0)

tuned = DecisionTreeRegressor(max_depth=best_depth, random_state=RANDOM_STATE)
tuned.fit(X_train_full, y_train_full)

pred_full_train = tuned.predict(X_train_full)
pred_test = tuned.predict(X_test)

summary = pd.DataFrame([
    regression_report(y_train_full, pred_full_train, "train+val"),
    regression_report(y_test, pred_test, "test"),
])
summary

---
## 7) Feature importances

A regression tree provides a simple feature-importance heuristic based on the
total impurity reduction contributed by each feature across splits.

This isn't a causal statement, but it can be a useful sanity check.

---

In [ ]:
importances = tuned.feature_importances_
fi = pd.DataFrame({"feature": X.columns, "importance": importances}).sort_values("importance", ascending=False)
fi

In [ ]:
plt.figure()
plt.barh(fi["feature"].iloc[::-1], fi["importance"].iloc[::-1])
plt.xlabel("Importance")
plt.title("Decision tree feature importances")
plt.show()

---
## 8) Inspecting predictions on a few examples

Finally, let's compare a handful of test samples with their predicted values.
This can help you get an intuition for model error in the original units.

---

In [ ]:
preview = X_test.copy()
preview["y_true"] = y_test.values
preview["y_pred"] = pred_test
preview["abs_error"] = np.abs(preview["y_true"] - preview["y_pred"])

preview.sort_values("abs_error", ascending=False).head(10)

---
## 9) Takeaways

- Decision trees are **interpretable** and easy to train, but can **overfit** quickly.
- A simple hyperparameter like `max_depth` often makes a huge difference.
- Validation-based tuning is a practical baseline; for production use, prefer cross-validation.
- Ensembles (Random Forests / Gradient Boosting) often outperform a single tree, especially on noisy tabular data.

---